# Rubin DP2 × Gaia AGN DRW Variability Analysis

**Goals**
1. Cross-match Gaia AGN catalog with Rubin DP2 difference-imaging sources
2. Select ≥ 200 per-band lightcurves (≥ 30-day baseline, densest cadence)
3. Remove flagged detections
4. Fit a Damped Random Walk (DRW) model with EzTaoX to each lightcurve
5. Identify the band most variable on ~10-day timescales
6. Identify the most variable AGN per band (bands with > 10 LCs)
7. Plot lightcurve + DRW fit for each top AGN
8. Assess physical vs. instrumental origin of variability

## 0 · Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import AutoMinorLocator
from scipy.stats import spearmanr

import lsdb

# ── Catalog paths ──────────────────────────────────────────────────────────────
DP2_COLLECTION = "/sdf/data/rubin/shared/lsdb_commissioning/hats/v30_0_6/dia_object_collection"
# Gaia DR3 AGN/QSO candidates in HATS format.
# If this URL fails, check https://data.lsdb.io/hats/ for the correct path.
GAIA_AGN_URL   = "https://data.lsdb.io/hats/gaia_dr3_agn"

# ── Analysis thresholds ────────────────────────────────────────────────────────
XMATCH_RADIUS_ARCSEC = 1.0    # Gaia → DiaObject match radius
MIN_OBS_PER_LC       = 10     # minimum detections per (object, band) LC
MIN_SPAN_DAYS        = 30     # minimum time baseline
TARGET_N_LCS         = 200    # target sample size (densest-cadence selection)
MIN_LCS_FOR_BAND     = 10     # bands need ≥ this many LCs for ranking
SF_LAG_DAYS          = 10.0   # lag at which to evaluate structure function

# ── Expected quality-flag column names (Rubin DP2 DiaSource) ─────────────────
BAD_FLAG_COLS = [
    "pixelFlags_bad",
    "pixelFlags_cr",
    "pixelFlags_crCenter",
    "pixelFlags_saturated",
    "pixelFlags_saturatedCenter",
    "pixelFlags_edge",
    "pixelFlags_offimage",
    "pixelFlags_interpolatedCenter",
]

BANDS       = list('ugrizy')
BAND_COLORS = {'u':'#7B2D8B','g':'#1A9E49','r':'#D92525',
               'i':'#E8821A','z':'#1B6FC2','y':'#7B4A14'}
print("Setup complete.")

### 0.1 · EzTaoX API probe
Identify which API pattern is available so the fitting wrapper below can choose the correct path.

In [ ]:
import eztaox
print(f"eztaox version : {getattr(eztaox, '__version__', 'unknown')}")
print(f"Top-level names: {[a for a in dir(eztaox) if not a.startswith('_')]}")

# Detect which sub-modules / functions exist
_HAS_TS_MODULE   = False
_HAS_FIT_FN      = False
_HAS_DRW_CLASS   = False

try:
    from eztaox.ts import drw_fit as _eztaox_drw_fit, pred_lc as _eztaox_pred_lc
    _HAS_TS_MODULE = True
    print("✓ eztaox.ts.drw_fit / pred_lc found")
except ImportError:
    print("  eztaox.ts not available")

if not _HAS_TS_MODULE:
    try:
        _eztaox_fit_fn = getattr(eztaox, 'fit_drw', None) or getattr(eztaox, 'drw_fit', None)
        if _eztaox_fit_fn is not None:
            _HAS_FIT_FN = True
            print("✓ eztaox.fit_drw / drw_fit function found")
    except Exception:
        pass

if not _HAS_TS_MODULE and not _HAS_FIT_FN:
    try:
        _EztaoxDRW = getattr(eztaox, 'DRWModel', None) or getattr(eztaox, 'DRW', None)
        if _EztaoxDRW is not None:
            _HAS_DRW_CLASS = True
            print("✓ eztaox DRW class found")
    except Exception:
        pass

if not any([_HAS_TS_MODULE, _HAS_FIT_FN, _HAS_DRW_CLASS]):
    print("⚠ Could not detect eztaox API — edit fit_drw_model() below manually")

### 0.2 · DRW fitting helpers

In [ ]:
def fit_drw_model(t, flux, flux_err):
    """
    Fit a DRW (CAR(1)) model to a lightcurve.

    Tries three API patterns in order:
      1. eztaox.ts.drw_fit  (EzTao-style)
      2. eztaox.fit_drw / eztaox.drw_fit  (top-level function)
      3. eztaox.DRWModel / eztaox.DRW     (class-based)

    Returns dict with keys: sigma, tau, t0, success
    sigma : DRW amplitude (same units as flux)
    tau   : correlation timescale (days)
    """
    if len(t) < 5:
        return {'success': False, 'error': 'too few points'}

    t0  = float(t.min())
    tn  = t - t0                          # time relative to first observation
    mu  = float(np.median(flux))
    yn  = flux - mu                        # mean-subtract
    ye  = flux_err.copy()

    # Pattern 1 – eztaox.ts
    if _HAS_TS_MODULE:
        try:
            best, _ = _eztaox_drw_fit(tn, yn, ye)
            sigma, tau = float(best[0]), float(best[1])
            return dict(sigma=abs(sigma), tau=max(abs(tau), 0.01),
                        t0=t0, mu=mu, api='ts', success=True)
        except Exception as e:
            pass

    # Pattern 2 – top-level function
    if _HAS_FIT_FN:
        try:
            result = _eztaox_fit_fn(tn, yn, ye)
            sigma = float(getattr(result, 'sigma', result[0]))
            tau   = float(getattr(result, 'tau',   result[1]))
            gp    = getattr(result, 'gp', None)
            return dict(sigma=abs(sigma), tau=max(abs(tau), 0.01),
                        t0=t0, mu=mu, gp=gp, api='func', success=True)
        except Exception as e:
            pass

    # Pattern 3 – class
    if _HAS_DRW_CLASS:
        try:
            model = _EztaoxDRW()
            model.fit(tn, yn, ye)
            sigma = float(getattr(model, 'sigma_', getattr(model, 'sigma', np.nan)))
            tau   = float(getattr(model, 'tau_',   getattr(model, 'tau',   np.nan)))
            return dict(sigma=abs(sigma), tau=max(abs(tau), 0.01),
                        t0=t0, mu=mu, gp=model, api='class', success=True)
        except Exception as e:
            return {'success': False, 'error': str(e)}

    return {'success': False, 'error': 'no working eztaox API found'}


def predict_drw_model(fit, t_obs, flux_obs, flux_err_obs, t_pred):
    """
    Compute DRW posterior mean and std at t_pred.
    Returns (mu_pred, std_pred) arrays, or (None, None) on failure.
    """
    if fit is None or not fit.get('success', False):
        return None, None

    t0     = fit.get('t0', 0.0)
    mu_lc  = fit.get('mu', 0.0)
    tn_obs  = t_obs  - t0
    tn_pred = t_pred - t0
    yn_obs  = flux_obs - mu_lc

    try:
        if fit.get('api') == 'ts':
            params = [fit['sigma'], fit['tau']]
            mu_p, std_p = _eztaox_pred_lc(tn_obs, yn_obs, flux_err_obs, params, tn_pred)
            return mu_p + mu_lc, std_p

        gp = fit.get('gp')
        if gp is None:
            return None, None

        # celerite2-style GP
        try:
            gp.compute(tn_obs, flux_err_obs)
            mu_p, var_p = gp.predict(yn_obs, tn_pred, return_var=True)
            return mu_p + mu_lc, np.sqrt(np.abs(var_p))
        except TypeError:
            mu_p = gp.predict(tn_pred)
            return np.atleast_1d(mu_p) + mu_lc, np.full(len(t_pred), fit['sigma'])

    except Exception:
        return None, None


def sf_at_lag(sigma, tau, lag):
    """DRW structure function: SF(Δt) = σ√2 · √(1 − exp(−Δt/τ))."""
    return sigma * np.sqrt(2.0 * (1.0 - np.exp(-lag / max(tau, 1e-6))))


print("Helper functions defined.")

## 1 · Load and Explore Catalogs

In [ ]:
print(f"Loading Gaia AGN catalog from:\n  {GAIA_AGN_URL}")
try:
    gaia_agn = lsdb.open_catalog(GAIA_AGN_URL)
    print(f"  rows    : {gaia_agn.hc_structure.catalog_info.total_rows:,}")
    print(f"  columns : {sorted(gaia_agn.columns)[:12]} ...")
except Exception as err:
    print(f"\n⚠ Failed to load {GAIA_AGN_URL}")
    print(f"  Error: {err}")
    print("  Try one of these alternatives:")
    for alt in [
        "https://data.lsdb.io/hats/gaia_dr3/qso_candidates",
        "https://data.lsdb.io/hats/gaia_agn",
        "https://data.lsdb.io/hats/gaia_dr3_qso",
    ]:
        print(f"    lsdb.open_catalog('{alt}')")
    raise

In [ ]:
print(f"Loading DIA Object catalog...")
dia_object = lsdb.open_catalog(f"{DP2_COLLECTION}/DiaObject")
print(f"  rows    : {dia_object.hc_structure.catalog_info.total_rows:,}")
print(f"  columns : {sorted(dia_object.columns)}")

In [ ]:
print(f"Loading DIA Source catalog...")
dia_source = lsdb.open_catalog(f"{DP2_COLLECTION}/DiaSource")
print(f"  rows    : {dia_source.hc_structure.catalog_info.total_rows:,}")
print(f"  columns : {sorted(dia_source.columns)}")

## 2 · Cross-Match Gaia AGN × Rubin DIA Objects

In [ ]:
print(f"Cross-matching Gaia AGN × DIA Objects (r < {XMATCH_RADIUS_ARCSEC}\")...")
xmatch = gaia_agn.crossmatch(
    dia_object,
    n_neighbors=1,
    radius_arcsec=XMATCH_RADIUS_ARCSEC,
)
xmatch_df = xmatch.compute()
print(f"  Matched pairs : {len(xmatch_df):,}")
print(f"  Columns       : {list(xmatch_df.columns)}")
xmatch_df.head(3)

In [ ]:
# Resolve column name for diaObjectId (may be suffixed after crossmatch)
obj_id_candidates = [c for c in xmatch_df.columns
                     if 'diaobjectid' in c.lower() or c == 'objectId']
if not obj_id_candidates:
    raise RuntimeError(f"Cannot find diaObjectId column; available: {list(xmatch_df.columns)}")
OBJ_ID_XMATCH = obj_id_candidates[0]
print(f"Using '{OBJ_ID_XMATCH}' as the matched DIA Object ID column")

matched_oids = set(xmatch_df[OBJ_ID_XMATCH].values)
print(f"Unique matched DIA Objects : {len(matched_oids):,}")

## 3 · Load DIA Sources for Matched AGN

In [ ]:
print("Loading DIA Sources for matched objects...")
print("  Strategy: compute DIA Source catalog and filter by matched diaObjectId.")
print("  (HATS spatial partitioning limits I/O to relevant sky pixels.)")

# The DiaSource catalog shares HATS partitioning with DiaObject, so loading
# only the partitions covering our matched sky positions is automatic.
try:
    # Preferred: join within LSDB before computing (avoids loading entire catalog)
    joined = xmatch.join(
        dia_source,
        left_on=OBJ_ID_XMATCH,
        right_on='diaObjectId',
        suffixes=('_obj', '_src'),
    )
    dia_src_raw = joined.compute()
    print(f"  LSDB join succeeded: {len(dia_src_raw):,} sources")
except Exception as join_err:
    print(f"  LSDB join failed ({join_err}), falling back to compute-then-filter...")
    dia_src_all = dia_source.compute()
    obj_col = next((c for c in dia_src_all.columns
                    if 'diaobjectid' in c.lower()), None)
    if obj_col is None:
        raise RuntimeError(f"Cannot find diaObjectId in DiaSource; columns: {list(dia_src_all.columns)}")
    dia_src_raw = dia_src_all[dia_src_all[obj_col].isin(matched_oids)].copy()
    print(f"  Compute+filter: {len(dia_src_raw):,} sources")

print(f"  Columns: {list(dia_src_raw.columns)}")
dia_src_raw.head(3)

In [ ]:
# ── Auto-detect and standardize column names ──────────────────────────────────
cols = list(dia_src_raw.columns)

def pick_col(candidates, available, label):
    for c in candidates:
        if c in available:
            print(f"  {label}: {c}")
            return c
    raise RuntimeError(f"Cannot find {label} column; tried {candidates}; available: {available}")

TIME_COL  = pick_col(['midpointMjdTai','midPointTai','midpointTai','visitTime','expMidptMJD'], cols, 'time')
BAND_COL  = pick_col(['band','filterName','filter','bandpass'], cols, 'band')
OID_COL   = pick_col(['diaObjectId','objectId','dia_object_id',
                       'diaObjectId_src','diaObjectId_obj'], cols, 'diaObjectId')
FLUX_COL  = pick_col(['psfFlux','psf_flux','flux','ap_flux'], cols, 'flux')
FERR_COL  = pick_col(['psfFluxErr','psf_flux_err','fluxErr','flux_err','ap_flux_err'], cols, 'flux_err')

src = dia_src_raw.rename(columns={
    TIME_COL: 'mjd',
    BAND_COL: 'band',
    OID_COL:  'diaObjectId',
    FLUX_COL: 'flux',
    FERR_COL: 'flux_err',
}).copy()

print(f"\nTotal raw DIA sources for matched AGN: {len(src):,}")

## 4 · Quality Flag Filtering

In [ ]:
n_before = len(src)

present_flag_cols = [c for c in BAD_FLAG_COLS if c in src.columns]
print(f"Pixel flag columns found: {present_flag_cols}")

if present_flag_cols:
    # Exclude sources where any bad pixel flag is set
    bad = src[present_flag_cols].fillna(False).astype(bool).any(axis=1)
    src = src[~bad].copy()
    print(f"  After pixel-flag cut : {len(src):,} ({100*len(src)/n_before:.1f}% retained)")
elif 'flags' in src.columns:
    # Bitmask fallback: reject saturated (bit 9), CR (bit 3), bad (bit 0), edge (bit 4)
    BAD_BITS = (1 << 0) | (1 << 3) | (1 << 4) | (1 << 9)
    src = src[(src['flags'].fillna(0).astype(int) & BAD_BITS) == 0].copy()
    print(f"  After bitmask flag cut : {len(src):,} ({100*len(src)/n_before:.1f}% retained)")
else:
    print("  ⚠ No recognised flag columns — using all sources. Check column names.")

# Require finite, positive-error flux
n_before_finite = len(src)
src = src[
    np.isfinite(src['flux']) &
    np.isfinite(src['flux_err']) &
    (src['flux_err'] > 0)
].copy()
print(f"  After finite-flux cut  : {len(src):,} (removed {n_before_finite - len(src):,} NaN/inf rows)")

print(f"\nBand distribution after filtering:")
print(src.groupby('band').size().to_frame('n_sources').T)

## 5 · Build Per-Band Lightcurves and Select Sample

In [ ]:
def lc_summary(grp):
    span  = grp['mjd'].max() - grp['mjd'].min()
    n     = len(grp)
    return pd.Series({
        'n_obs':        n,
        'span_days':    span,
        'cadence_days': span / max(n - 1, 1),   # mean time between obs
        'median_flux':  grp['flux'].median(),
        'flux_mad':     (grp['flux'] - grp['flux'].median()).abs().median(),
    })

lc_stats = (
    src.groupby(['diaObjectId', 'band'])
       .apply(lc_summary)
       .reset_index()
)

print(f"Total (object, band) groups: {len(lc_stats):,}")
print(lc_stats.groupby('band')['diaObjectId'].count().to_frame('n_lcs').T)

In [ ]:
# Apply baseline and minimum-observation cuts
lc_good = lc_stats[
    (lc_stats['n_obs']     >= MIN_OBS_PER_LC) &
    (lc_stats['span_days'] >= MIN_SPAN_DAYS)
].copy()

print(f"After ≥{MIN_OBS_PER_LC} obs + ≥{MIN_SPAN_DAYS}-day cut: {len(lc_good):,} LCs")
print(lc_good.groupby('band')['diaObjectId'].count().to_frame('n_lcs').T)

# "Densest cadence" selection: prefer LCs with shortest mean cadence
if len(lc_good) > TARGET_N_LCS:
    lc_selected = (
        lc_good
        .sort_values('cadence_days')   # ascending = densest first
        .head(TARGET_N_LCS)
        .reset_index(drop=True)
    )
    print(f"\nSelected top-{TARGET_N_LCS} densest-cadence LCs")
else:
    lc_selected = lc_good.reset_index(drop=True)
    print(f"\n{len(lc_selected)} qualifying LCs — fewer than target ({TARGET_N_LCS}); using all")

print(f"\nFinal sample band distribution:")
print(lc_selected.groupby('band')['diaObjectId'].count().to_frame('n_lcs').T)

## 6 · DRW Model Fitting (EzTaoX)

In [ ]:
try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kw): return x  # silent fallback

drw_rows   = []
fit_store  = {}
n_failed   = 0

for _, row in tqdm(lc_selected.iterrows(), total=len(lc_selected), desc="DRW fits"):
    oid  = row['diaObjectId']
    band = row['band']

    lc = (
        src[(src['diaObjectId'] == oid) & (src['band'] == band)]
        .sort_values('mjd')
    )

    t  = lc['mjd'].values.astype(float)
    y  = lc['flux'].values.astype(float)
    ye = lc['flux_err'].values.astype(float)

    fit = fit_drw_model(t, y, ye)

    if fit and fit['success']:
        sigma     = fit['sigma']
        tau       = fit['tau']
        sf10      = sf_at_lag(sigma, tau, SF_LAG_DAYS)
        med_flux  = abs(row['median_flux'])
        frac_sf10 = sf10 / med_flux if med_flux > 0 else np.nan

        drw_rows.append(dict(
            diaObjectId = oid,
            band        = band,
            sigma       = sigma,
            tau         = tau,
            sf10        = sf10,
            frac_sf10   = frac_sf10,
            n_obs       = row['n_obs'],
            span_days   = row['span_days'],
            median_flux = row['median_flux'],
        ))
        fit_store[(oid, band)] = fit
    else:
        n_failed += 1

drw_df = pd.DataFrame(drw_rows)
print(f"DRW fits: {len(drw_df):,} succeeded, {n_failed:,} failed")
drw_df.describe()

## 7 · Variability Analysis — 10-day Structure Function

In [ ]:
# Per-band summary (only bands meeting minimum LC count)
band_sf = (
    drw_df
    .groupby('band')['frac_sf10']
    .agg(median='median', mean='mean', std='std', n_lcs='count')
    .sort_values('median', ascending=False)
)

print("=" * 62)
print(f"SF({SF_LAG_DAYS:.0f} d) / |median flux|  —  per-band summary")
print("=" * 62)
print(band_sf.to_string())

# Bands with enough LCs for ranking
ranked_bands = band_sf[band_sf['n_lcs'] >= MIN_LCS_FOR_BAND]
most_variable_band = ranked_bands.index[0] if len(ranked_bands) > 0 else None
print(f"\n→ Most variable band on ~{SF_LAG_DAYS:.0f}-day scales: "
      f"{most_variable_band} "
      f"(median SF({SF_LAG_DAYS:.0f}d)/flux = "
      f"{ranked_bands.loc[most_variable_band,'median']:.4f})")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

valid = [b for b in BANDS if b in band_sf.index]
meds  = [band_sf.loc[b, 'median'] for b in valid]
stds  = [band_sf.loc[b, 'std']    for b in valid]
clrs  = [BAND_COLORS.get(b, '#888') for b in valid]

bars = ax.bar(valid, meds, yerr=stds, capsize=4,
              color=clrs, alpha=0.85, edgecolor='black', linewidth=0.7)
ax.set_xlabel("Band", fontsize=12)
ax.set_ylabel(f"Median SF({SF_LAG_DAYS:.0f} d) / |median flux|", fontsize=11)
ax.set_title(f"AGN fractional variability on ~{SF_LAG_DAYS:.0f}-day timescales", fontsize=12)
ax.yaxis.set_minor_locator(AutoMinorLocator())

for b, bar, med in zip(valid, bars, meds):
    ax.text(bar.get_x() + bar.get_width()/2, med + 0.0005,
            f'{med:.3f}', ha='center', va='bottom', fontsize=8)

# Annotate n per band
for b, bar in zip(valid, bars):
    n = int(band_sf.loc[b, 'n_lcs'])
    ax.text(bar.get_x() + bar.get_width()/2, -ax.get_ylim()[1]*0.04,
            f'n={n}', ha='center', va='top', fontsize=7, color='#444')

plt.tight_layout()
plt.savefig("band_variability_sf10.pdf", bbox_inches='tight', dpi=150)
plt.show()

## 8 · Most Variable AGN Per Band

In [ ]:
top_agn = {}   # band → Series (row from drw_df)

print(f"Most variable AGN per band (bands with ≥ {MIN_LCS_FOR_BAND} LCs):")
print("-" * 72)
for band in BANDS:
    sub = drw_df[drw_df['band'] == band].dropna(subset=['frac_sf10'])
    if len(sub) < MIN_LCS_FOR_BAND:
        print(f"  {band}: {len(sub)} LCs — skipping")
        continue
    best = sub.loc[sub['frac_sf10'].idxmax()]
    top_agn[band] = best
    print(f"  {band} ({len(sub):3d} LCs): "
          f"diaObjectId={best['diaObjectId']}  "
          f"σ={best['sigma']:.1f} nJy  "
          f"τ={best['tau']:.0f} d  "
          f"SF({SF_LAG_DAYS:.0f}d)/flux={best['frac_sf10']*100:.2f}%")

print("-" * 72)
top_agn_df = pd.DataFrame(top_agn).T.reset_index(names='band')
top_agn_df[['band','diaObjectId','sigma','tau','sf10','frac_sf10','n_obs','span_days']]

## 9 · Lightcurve + DRW Fit Plots for Top AGN

In [ ]:
N_PRED = 400  # prediction grid points

for band, best in top_agn.items():
    oid = best['diaObjectId']
    fit = fit_store.get((oid, band))

    lc = (
        src[(src['diaObjectId'] == oid) & (src['band'] == band)]
        .sort_values('mjd')
    )
    t   = lc['mjd'].values.astype(float)
    y   = lc['flux'].values.astype(float)
    ye  = lc['flux_err'].values.astype(float)

    t_grid      = np.linspace(t.min(), t.max(), N_PRED)
    mu_p, std_p = predict_drw_model(fit, t, y, ye, t_grid)

    col = BAND_COLORS.get(band, 'steelblue')
    fig, axes = plt.subplots(2, 1, figsize=(11, 6),
                             gridspec_kw={'height_ratios': [3, 1]}, sharex=True)

    # ── Top panel: lightcurve + DRW ───────────────────────────────────────────
    ax = axes[0]
    ax.errorbar(t, y, yerr=ye, fmt='o', color=col, ms=4,
                elinewidth=0.8, capsize=2, alpha=0.75, label='Observations', zorder=3)

    if mu_p is not None:
        ax.plot(t_grid, mu_p, color='black', lw=1.5, label='DRW mean', zorder=4)
        if std_p is not None:
            ax.fill_between(t_grid, mu_p - std_p, mu_p + std_p,
                            color='black', alpha=0.12, label='DRW ±1σ', zorder=2)
    else:
        ax.text(0.5, 0.95, '(GP prediction unavailable — check eztaox API)',
                transform=ax.transAxes, ha='center', va='top',
                fontsize=8, color='red')

    param_txt = (
        f"σ = {best['sigma']:.1f} nJy\n"
        f"τ = {best['tau']:.0f} d\n"
        f"SF({SF_LAG_DAYS:.0f}d)/flux = {best['frac_sf10']*100:.2f}%\n"
        f"N = {int(best['n_obs'])} epochs, "
        f"baseline = {best['span_days']:.0f} d"
    )
    ax.text(0.02, 0.97, param_txt, transform=ax.transAxes,
            fontsize=8, va='top', ha='left',
            bbox=dict(boxstyle='round,pad=0.35', fc='white', alpha=0.85))

    ax.set_ylabel("PSF flux (nJy)", fontsize=10)
    ax.set_title(f"Band {band}  —  Most Variable AGN  (diaObjectId = {oid})", fontsize=11)
    ax.legend(fontsize=9, loc='upper right')
    ax.yaxis.set_minor_locator(AutoMinorLocator())

    # ── Bottom panel: standardised residuals ─────────────────────────────────
    ax2 = axes[1]
    if mu_p is not None:
        # Interpolate DRW mean at observed times
        mu_at_t = np.interp(t, t_grid, mu_p)
        resid   = (y - mu_at_t) / ye
        ax2.axhline(0, color='black', lw=0.8, ls='--')
        ax2.axhspan(-3, 3, color='gray', alpha=0.08)
        ax2.scatter(t, resid, color=col, s=10, alpha=0.7)
        ax2.set_ylabel("(obs − DRW) / σ", fontsize=9)
        ax2.set_ylim(-6, 6)
    else:
        ax2.set_visible(False)

    ax2.set_xlabel("MJD", fontsize=10)
    ax2.xaxis.set_minor_locator(AutoMinorLocator())

    plt.tight_layout()
    fname = f"top_agn_band_{band}.pdf"
    plt.savefig(fname, bbox_inches='tight', dpi=150)
    print(f"  Saved {fname}")
    plt.show()

## 10 · Physical vs. Instrumental Variability Assessment

| Test | Physical expectation | Instrumental signature |
|------|---------------------|------------------------|
| Wavelength dependence | u > g > r > i > z > y (accretion-disk thermal fluctuations, shorter λ → inner disk → higher amplitude) | No clear chromatic trend, or *longer* λ more variable |
| DRW timescale τ | 100–300 d for luminous AGN | τ ≪ 10 d (readout noise) or τ ≫ 1000 d (unconstrained baseline) |
| Flux–variability correlation | Weak or absent (intrinsic variability is flux-independent at fixed luminosity) | Strong anti-correlation: faint sources noisier |
| Inter-band correlation | Correlated across bands with small lag (reprocessing) | Uncorrelated or correlated with observing conditions |

In [ ]:
# ── 1. Wavelength dependence ──────────────────────────────────────────────────
EFFECTIVE_WAVELENGTH = {'u': 356, 'g': 470, 'r': 618, 'i': 750, 'z': 880, 'y': 1020}  # nm

band_medians = drw_df.groupby('band')['frac_sf10'].median()
plotbands    = [b for b in BANDS if b in band_medians.index]
wav          = [EFFECTIVE_WAVELENGTH[b] for b in plotbands]
vals         = [band_medians[b] for b in plotbands]

fig, ax = plt.subplots(figsize=(6, 4))
for b, w, v in zip(plotbands, wav, vals):
    ax.scatter(w, v, color=BAND_COLORS.get(b,'gray'), s=120,
               zorder=3, label=b)
    ax.annotate(b, (w, v), textcoords='offset points', xytext=(5, 4),
                fontsize=10, fontweight='bold',
                color=BAND_COLORS.get(b,'gray'))

# Simple linear fit to assess slope direction
if len(wav) >= 2:
    z1 = np.polyfit(wav, vals, 1)
    ww = np.linspace(min(wav)-20, max(wav)+20, 100)
    ax.plot(ww, np.polyval(z1, ww), 'k--', lw=1, alpha=0.6,
            label=f'slope {z1[0]*1000:.2e} per nm')
    slope_phys = z1[0] < 0   # physical: variability decreases with wavelength

ax.set_xlabel("Effective wavelength (nm)", fontsize=11)
ax.set_ylabel(f"Median SF({SF_LAG_DAYS:.0f}d)/flux", fontsize=11)
ax.set_title("Variability wavelength dependence", fontsize=11)
ax.legend(fontsize=9, framealpha=0.8)
plt.tight_layout()
plt.savefig("variability_vs_wavelength.pdf", bbox_inches='tight', dpi=150)
plt.show()

print(f"Wavelength–variability slope: {'negative (u most variable → PHYSICAL)' if slope_phys else 'positive (y most variable → SUSPECT instrumental)'}")

In [ ]:
# ── 2. DRW timescale distribution ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
for band in BANDS:
    sub = drw_df[drw_df['band'] == band]['tau']
    if len(sub) < 3:
        continue
    ax.hist(np.log10(sub.clip(lower=0.1)), bins=20, alpha=0.5,
            color=BAND_COLORS.get(band,'gray'), label=band, density=True)

ax.axvspan(np.log10(100), np.log10(300), color='green', alpha=0.08,
           label='Physical AGN τ range (100–300 d)')
ax.axvline(np.log10(5),    color='red',   lw=1.2, ls='--', label='τ = 5 d (instrumental flag)')
ax.axvline(np.log10(1000), color='orange',lw=1.2, ls='--', label='τ = 1000 d (unconstrained)')

ax.set_xlabel("log₁₀(τ / days)", fontsize=11)
ax.set_ylabel("Density", fontsize=11)
ax.set_title("DRW timescale distribution per band", fontsize=11)
ax.legend(fontsize=8, framealpha=0.85)
plt.tight_layout()
plt.savefig("drw_tau_distribution.pdf", bbox_inches='tight', dpi=150)
plt.show()

n_short = (drw_df['tau'] < 5).sum()
n_long  = (drw_df['tau'] > 1000).sum()
print(f"τ < 5 d  (suspect instrumental) : {n_short} / {len(drw_df)} = {100*n_short/len(drw_df):.1f}%")
print(f"τ > 1000 d (unconstrained baseline): {n_long} / {len(drw_df)} = {100*n_long/len(drw_df):.1f}%")

In [ ]:
# ── 3. Variability amplitude vs. flux level ────────────────────────────────────
# Physical: intrinsic variability has weak dependence on flux (at fixed z)
# Instrumental: faint objects inflate σ via photon noise; shows as anti-correlation

fig, ax = plt.subplots(figsize=(7, 5))
for band in BANDS:
    sub = drw_df[(drw_df['band'] == band) &
                 drw_df['frac_sf10'].notna() &
                 (drw_df['median_flux'] > 0)]
    if len(sub) < 3:
        continue
    ax.scatter(sub['median_flux'], sub['frac_sf10'],
               color=BAND_COLORS.get(band,'gray'), s=18,
               alpha=0.6, label=band)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel("Median PSF flux (nJy, log scale)", fontsize=11)
ax.set_ylabel(f"SF({SF_LAG_DAYS:.0f}d) / flux (log scale)", fontsize=11)
ax.set_title("Variability fraction vs. source brightness", fontsize=11)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("variability_vs_flux.pdf", bbox_inches='tight', dpi=150)
plt.show()

valid = drw_df.dropna(subset=['frac_sf10','median_flux'])
valid = valid[valid['median_flux'] > 0]
rho, pval = spearmanr(np.log10(valid['median_flux']), np.log10(valid['frac_sf10']))
print(f"Spearman ρ(log flux, log SF10/flux) = {rho:.3f}  (p = {pval:.2e})")
if rho < -0.4:
    print("  Strong anti-correlation → noise floor likely inflating variability for faint sources.")
    print("  Consider applying a minimum S/N cut (e.g. median flux > 5 × median flux_err).")
else:
    print("  Weak flux–variability correlation → variability consistent with intrinsic AGN origin.")

In [ ]:
# ── 4. Inter-band variability correlation for the same AGN ────────────────────
# Physical: variability amplitude correlated across bands (reprocessing + disk)
# Instrumental: decorrelated because independent readout noise per band

pivot = drw_df.pivot_table(
    index='diaObjectId', columns='band', values='frac_sf10'
)

# Choose two bands with the most overlap
band_overlap = {
    (b1, b2): pivot[[b1, b2]].dropna().shape[0]
    for b1 in BANDS for b2 in BANDS
    if b1 < b2 and b1 in pivot.columns and b2 in pivot.columns
}

if band_overlap:
    best_pair = max(band_overlap, key=band_overlap.get)
    b1, b2 = best_pair
    sub_p = pivot[[b1, b2]].dropna()
    print(f"Best band pair for inter-band correlation: {b1} vs {b2} ({len(sub_p)} common AGN)")

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(sub_p[b1], sub_p[b2],
               alpha=0.6, s=20,
               color='#2C7BB6')
    lo = min(sub_p.min().min(), sub_p.min().min())
    hi = max(sub_p.max().max(), sub_p.max().max())
    ax.plot([lo, hi], [lo, hi], 'k--', lw=1, label='1:1')
    ax.set_xlabel(f"SF({SF_LAG_DAYS:.0f}d)/flux  [{b1}]", fontsize=11)
    ax.set_ylabel(f"SF({SF_LAG_DAYS:.0f}d)/flux  [{b2}]", fontsize=11)
    ax.set_title(f"{b1}-band vs {b2}-band variability\n(same AGN, {len(sub_p)} objects)", fontsize=11)

    rho_ib, p_ib = spearmanr(sub_p[b1], sub_p[b2])
    ax.text(0.05, 0.95, f"ρ = {rho_ib:.3f}", transform=ax.transAxes,
            fontsize=11, va='top')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(f"interband_corr_{b1}_{b2}.pdf", bbox_inches='tight', dpi=150)
    plt.show()

    if rho_ib > 0.5:
        print(f"  ρ = {rho_ib:.3f} → strong inter-band correlation — variability is PHYSICAL")
    elif rho_ib > 0.2:
        print(f"  ρ = {rho_ib:.3f} → moderate correlation — mix of physical and noise")
    else:
        print(f"  ρ = {rho_ib:.3f} → weak correlation — noise may dominate; review S/N cuts")
else:
    print("Insufficient multi-band overlap for inter-band correlation plot.")

## 11 · Summary

In [ ]:
print("=" * 70)
print(" SUMMARY")
print("=" * 70)
print(f"  Gaia AGN matched to Rubin DP2 DIA Objects : {len(xmatch_df):,}")
print(f"  DIA sources after quality filtering        : {len(src):,}")
print(f"  Lightcurves in final sample                : {len(lc_selected):,}")
print(f"  Successful DRW fits                        : {len(drw_df):,}")
print()
print(f"  Band most variable on ~{SF_LAG_DAYS:.0f}-day scales     : {most_variable_band}")
print()
print(f"  Most variable AGN per band (SF({SF_LAG_DAYS:.0f}d)/flux):")
for _, row in top_agn_df.iterrows():
    print(f"    {row['band']} : diaObjectId={row['diaObjectId']}  "
          f"σ={row['sigma']:.1f} nJy  τ={row['tau']:.0f} d  "
          f"SF10/flux={row['frac_sf10']*100:.2f}%")

print()
print("  Physical vs. Instrumental assessment:")
if 'slope_phys' in dir():
    print(f"    Wavelength trend    : {'physical (blue > red)' if slope_phys else 'atypical — investigate'}")
print(f"    τ < 5 d (suspicious): {(drw_df['tau'] < 5).sum()} / {len(drw_df)} LCs")
print(f"    Flux–variability ρ  : {rho:.3f} ({'noise-dominated' if rho < -0.4 else 'intrinsic-dominated'})")
if 'rho_ib' in dir():
    print(f"    Inter-band ρ ({b1},{b2})  : {rho_ib:.3f} ({'physical' if rho_ib > 0.5 else 'mixed/noise'})")
print("=" * 70)